In [ ]:
# Install mode selector
INSTALL_MODE = "dev"  # "pypi" or "dev"
REPO_URL = "https://github.com/HNXJ/jaxfne.git"
BRANCH = "dev"

import sys
import subprocess

if 'google.colab' in sys.modules or INSTALL_MODE == "pypi":
    if INSTALL_MODE == "pypi":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "jaxfne"])
    elif INSTALL_MODE == "dev":
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO_URL}@{BRANCH}"])


In [ ]:
# Canonical imports and environment receipt
import jaxfne as jtfne

import inspect
import importlib
import json
pkgutil_import = importlib.import_module("pkgutil")
from pathlib import Path

import jax
import numpy as np


In [ ]:
# API Inspection
print("jaxfne version:", jtfne.__version__)
print("jaxfne file:", jtfne.__file__)
print("jax version:", jax.__version__)
print("jax devices:", jax.devices())

public_api_names = dir(jtfne)
print("public API names length:", len(public_api_names))

# Check required names
REQUIRED_DELTA_NAMES = [
    "SanityDeltaConfig",
    "SanityDeltaModel",
    "HierarchicalOddballParadigm",
    "BehaviorGate",
    "BackupState",
    "TaskEpisode",
    "Manifest",
]

delta_names_found = {}
for name in REQUIRED_DELTA_NAMES:
    found = name in public_api_names
    delta_names_found[name] = found
    print(f"Name {name} found:", found)

# Check modules
has_sanity_delta = False
try:
    import jaxfne.sanity_delta
    has_sanity_delta = True
except ImportError:
    pass

has_sanity_runtime = False
try:
    import jaxfne.sanity_runtime
    has_sanity_runtime = True
except ImportError:
    pass

has_core = False
try:
    import jaxfne.core
    has_core = True
except ImportError:
    pass

print("has_sanity_delta:", has_sanity_delta)
print("has_sanity_runtime:", has_sanity_runtime)
print("has_core:", has_core)


In [ ]:
# Orientation report builder
# Resolve path relative to repo root
cwd = Path.cwd()
if cwd.name == "tutorials":
    repo_root = cwd.parent
else:
    repo_root = cwd

has_all_deltas = has_sanity_delta and has_sanity_runtime and all(delta_names_found.values())
selected_path = "v032_delta_full_smoke" if has_all_deltas else "v031_stable_smoke"

orientation_report = {
    "jaxfne_version": jtfne.__version__,
    "jaxfne_file": jtfne.__file__,
    "has_sanity_delta": has_sanity_delta,
    "has_sanity_runtime": has_sanity_runtime,
    "delta_names": delta_names_found,
    "selected_path": selected_path,
    "truth_mode": "truth_safe_unverified",
    "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
    "biological_learning_claim": False,
}

out_dir = repo_root / "outputs" / "v0333_colab_gemini_orientation"
out_dir.mkdir(parents=True, exist_ok=True)

with open(out_dir / "orientation_report.json", "w") as f:
    json.dump(orientation_report, f, allow_nan=False, indent=2)

print("Saved orientation report.")
print(json.dumps(orientation_report, indent=2))


In [ ]:
# Reflection text
if selected_path == "v031_stable_smoke":
    print("Installed jaxfne is stable/PyPI path. SanityDelta APIs are unavailable. I will not invent them. I will run a construct/simulate smoke only.")
elif selected_path == "v032_delta_full_smoke":
    print("Installed jaxfne exposes SanityDelta and sanity_runtime. I will run the full-mode delta smoke using package APIs.")


In [ ]:
# v0.3.31-compatible smoke
if selected_path == "v031_stable_smoke":
    cfg = jtfne.suite2_four_celltype_config(seed=0, duration_ms=1000.0, dt_ms=0.1)
    model = jtfne.construct(cfg)
    signals = jtfne.simulate(model, duration_ms=1000.0, dt_ms=0.1, seed=0)
    
    vm = signals.get("vm")
    spk = signals.get("spk")
    
    assert vm is not None and spk is not None
    assert np.isfinite(vm).all()
    assert np.isfinite(spk).all()
    
    v031_manifest = {
        "jaxfne_version": jtfne.__version__,
        "smoke_path": "v031_stable_smoke",
        "vm_shape": list(vm.shape),
        "spk_shape": list(spk.shape),
        "vm_min": float(vm.min()),
        "vm_max": float(vm.max()),
        "spk_sum": float(spk.sum()),
        "truth_mode": "truth_safe_unverified",
    }
    
    with open(out_dir / "v031_smoke_manifest.json", "w") as f:
        json.dump(v031_manifest, f, allow_nan=False, indent=2)
    print("v0.3.31 smoke verification completed successfully.")
else:
    print("Skipping v0.3.31 stable smoke path.")


In [ ]:
# v0.3.32-alpha delta smoke
if selected_path == "v032_delta_full_smoke":
    cfg = jtfne.SanityDeltaConfig.hierarchical_global_local_oddball(
        runtime_mode="full",
        seed=0,
        duration_ms=1000.0,
        dt_ms=0.1,
    )
    paradigm = cfg.make_paradigm()
    model = cfg.construct()
    gate = paradigm.make_fixation_gate()
    model = model.enable_plasticity()
    backup = model.initialize_backup(paradigm=paradigm, history_ms=1000.0)

    episode = model.run_task(
        paradigm=paradigm,
        gate=gate,
        backup=backup,
        runtime_mode="full",
    )
    episode = episode.probe(
        readouts=("spk", "vm", "source", "lfp_proxy", "csd_proxy", "eeg_proxy", "meg_proxy")
    )
    validation = episode.validate(
        checks=(
            "finite_outputs",
            "strict_json",
            "backup_resume_equivalence",
            "proxy_safe_readout_names",
            "truth_gates_preserved",
        )
    )
    
    delta_out_dir = out_dir / "v032_delta_smoke"
    episode.export(
        artifact_dir=str(delta_out_dir),
        strict_json=True,
    )
    
    vm = np.asarray(episode.signals["vm"])
    spk = np.asarray(episode.signals["spk"])
    
    assert np.isfinite(vm).all()
    assert np.isfinite(spk).all()
    
    print("v0.3.32-alpha delta smoke verification completed successfully.")
    print("Validation status:", validation)
else:
    print("Skipping v0.3.32 delta smoke path.")


In [ ]:
# Final Summary
print("=== FINAL SUMMARY ===")
print("selected_path:", selected_path)
print("version:", jtfne.__version__)

if selected_path == "v032_delta_full_smoke":
    print("runtime_mode: full")
    vm = np.asarray(episode.signals["vm"])
    spk = np.asarray(episode.signals["spk"])
    print("vm shape:", vm.shape)
    print("spk shape:", spk.shape)
    print("finite status: True")
    print("spike count:", float(spk.sum()))
    print("generated files under outputs/v0333_colab_gemini_orientation/v032_delta_smoke/")
    print("next action suggestion: Use dev branch and sanity_delta / sanity_runtime for hierarchical global-local oddball simulations.")
elif selected_path == "v031_stable_smoke":
    print("vm shape:", vm.shape)
    print("spk shape:", spk.shape)
    print("finite status: True")
    print("spike count:", float(spk.sum()))
    print("generated files: outputs/v0333_colab_gemini_orientation/v031_smoke_manifest.json")
    print("next action suggestion: Update to dev branch for advanced hierarchical oddball features.")
